# 00 · Download qualifying data (2018–2025)

Downloads every qualifying session with [FastF1](https://docs.fastf1.dev) and saves one parquet file per session to Google Drive.

**Rate limits:** FastF1 enforces hard API limits (Jolpica 200 calls/hour, all APIs 500 calls/hour), so eight seasons will not finish in one run. When the limit is hit the run stops cleanly. Wait about an hour and run the download cell again: finished sessions are skipped.

**Manual data:** sessions with no times in any API (currently 2025 Miami) use `manual_data/quali_times.csv`, copied from the official formula1.com results. Those sessions show `QTimeSource = manual`.


In [4]:
# @title Setup: mount Drive, get the latest code, install FastF1
REPO_URL = "https://github.com/khushi075/f1-race-sim.git"  # @param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/f1sim_data"  # @param {type:"string"}

import os, sys
from google.colab import drive

# Python 3.12+ compatibility patch for legacy packages or magics still using 'imp'
try:
    import imp
except ImportError:
    import types, importlib
    imp = types.ModuleType('imp')
    imp.reload = importlib.reload
    sys.modules['imp'] = imp

drive.mount("/content/drive")

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "fastf1>=3.9,<4"

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["F1SIM_DATA_DIR"] = DATA_DIR

%load_ext autoreload
%autoreload 2
print("Code:", REPO_DIR, "| Data:", DATA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.
/content/f1-race-sim
Code: /content/f1-race-sim | Data: /content/drive/MyDrive/f1sim_data


In [5]:
# @title Download (rerun this cell until nothing is left)
START_YEAR = 2018  # @param {type:"integer"}
END_YEAR = 2025  # @param {type:"integer"}

import fastf1
from f1sim.paths import get_paths
from f1sim.ingest import ingest_seasons

fastf1.set_log_level("WARNING")
paths = get_paths()
summary = ingest_seasons(range(START_YEAR, END_YEAR + 1), paths)

--- 2018 ---
--- 2019 ---
--- 2020 ---
--- 2021 ---
--- 2022 ---
--- 2023 ---
--- 2024 ---
--- 2025 ---


INFO:fastf1.fastf1.core:Loading data for Miami Grand Prix - Qualifying [v3.9.2]
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
INFO:fastf1.api:Fetching session info data...
INFO:fastf1.f1account:Signing in to Formula 1 account...
transport    WARNING 	Failed to authenticate with Formula 1 account, continuing without authentication: No F1 account credentials available. Set the environment variables F1_USERNAME and F1_PASSWORD.
DEBUG:fastf1.fastf1.req:Falling back to https://livetiming-mirror.fastf1.dev/static/2025/2025-05-04_Miami_Grand_Prix/2025-05-03_Qualifying/SessionInfo.jsonStream
logger      WARNING 	Failed to load session info data!
DEBUG:fastf1.fastf1.core:Traceback for failure in session info data
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/fastf1/core.py", line 1457, in _load_session_inf

saved  2025 R06 Miami Grand Prix  (manual)
Saved this run:     1
Already on disk:    172
Skipped (future):   0
Failed:             0


In [6]:
# @title Check what's on disk
from f1sim.ingest import load_quali

q = load_quali(paths)
sessions = q.drop_duplicates(["Year", "Round"])
coverage = (
    sessions.groupby("Year")
    .agg(Sessions=("Round", "size"))
    .join(sessions.groupby("Year")["QTimeSource"].value_counts().unstack(fill_value=0))
)
coverage

,Sessions,manual,results_api
Year,,,
2018,21,0,21
2019,21,0,21
2020,17,0,17
2021,22,0,22
2022,22,0,22
2023,22,0,22
2024,24,0,24
2025,24,1,23


In [7]:
# @title Teams with more than one name in a season (grouped by TeamId in the analysis)
names = q.groupby(["Year", "TeamId"])["TeamName"].unique()
names[names.map(len) > 1]

,,TeamName
Year,TeamId,


In [8]:
print("Missing TeamId:", q["TeamId"].isna().sum(), "| blank:", (q["TeamId"] == "").sum())
q.groupby("Year")["TeamId"].nunique()

Missing TeamId: 0 | blank: 0


,TeamId
Year,
2018,10
2019,10
2020,10
2021,11
2022,10
2023,10
2024,10
2025,10
